In [ ]:
import random
import pickle
import os

# =============================================================================
# 1. FASTA reader
# =============================================================================
data_dir = "data_and_cache"
fasta_path = os.path.join(data_dir, "mtor_referans_31.fasta")

print(f"Target FASTA file: {fasta_path}\n")

def read_fasta(file_path):
    gene_dict = {}
    current_gene_name = None
    current_sequence = []

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File '{os.path.basename(file_path)}' not found in '{data_dir}'.")

    # A FASTA record can span several lines, so sequence lines are buffered
    # until the next '>' header (or EOF) closes off the current gene.
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            stripped_line = line.strip()
            if not stripped_line:
                continue

            if ">" in stripped_line:
                if current_gene_name and current_sequence:
                    gene_dict[current_gene_name] = "".join(current_sequence).upper()

                raw_header = stripped_line.split(">")[-1].strip()
                current_gene_name = raw_header.split()[0] if raw_header else f"Gene_{len(gene_dict)+1}"
                current_sequence = []
            else:
                clean_dna = "".join(c for c in stripped_line if c.isalpha())
                if clean_dna:
                    current_sequence.append(clean_dna)

        if current_gene_name and current_sequence:
            gene_dict[current_gene_name] = "".join(current_sequence).upper()

    return gene_dict

gene_pool_dict = read_fasta(fasta_path)
gene_names = list(gene_pool_dict.keys())
all_gene_sequences = list(gene_pool_dict.values())

print("=================================================")
print("mTOR FASTA parsing report")
print("=================================================")
print(f"Parsed {len(gene_pool_dict)} genes from the reference file.\n")

if len(gene_pool_dict) > 0:
    for i in range(min(5, len(gene_names))):
        print(f" {i+1}. Gene: {gene_names[i]:<15} | Length: {len(all_gene_sequences[i])} bp")
print("=================================================\n")

if len(gene_pool_dict) == 0:
    print("Warning: no genes were parsed. Check the FASTA file before generating the population.")


# =============================================================================
# 2. Haploid pool generator
# =============================================================================
def build_haploid_pool(gene_sequences, count, group_type="control"):
    pool = []
    for i in range(count):
        individual_genes = []
        for dna in gene_sequences:
            # One random candidate position per 100 bp window (polymorphic/benign)
            # and per 200 bp window (pathogenic). Only a subset of candidates is
            # actually mutated below, so windows just cap how spread out sites are.
            polymorphic_positions = [random.randint(100*j, 100*(j+1)-1) for j in range(len(dna)//100)]
            pathogenic_positions = [random.randint(200*j, 200*(j+1)-1) for j in range(len(dna)//200)]
            dna_list = list(dna)

            # 40% of polymorphic candidates are mutated for every individual,
            # regardless of group, since these are not disease-relevant.
            for pos in random.choices(polymorphic_positions, k=(len(polymorphic_positions)*2)//5):
                if pos < len(dna_list):
                    dna_list[pos] = random.choice(list({'A','T','C','G'}.difference({dna_list[pos]})))

            # Patients get a higher pathogenic mutation rate (30%) than controls
            # (25%) -- this rate gap is the actual control/patient signal.
            rate = len(pathogenic_positions)//4 if group_type == "control" else (len(pathogenic_positions)*3)//10
            for pos in random.choices(pathogenic_positions, k=rate):
                if pos < len(dna_list):
                    dna_list[pos] = random.choice(list({'A','T','C','G'}.difference({dna_list[pos]})))

            individual_genes.append("".join(dna_list))
        pool.append(individual_genes)
    return pool

if len(gene_pool_dict) > 0:
    print("Generating haploid allele pools (this may take a while)...")
    control_haploid_pool = build_haploid_pool(all_gene_sequences, 800, group_type="control")
    patient_haploid_pool = build_haploid_pool(all_gene_sequences, 800, group_type="patient")

    haploid_dataset = {
        "controls_haploid": control_haploid_pool,
        "patients_haploid": patient_haploid_pool,
        "gene_names": gene_names
    }

    haploid_save_path = os.path.join(data_dir, "mTOR_HAPLOID_pool.pkl")
    with open(haploid_save_path, "wb") as f:
        pickle.dump(haploid_dataset, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"\n**************************************************")
    print(f"Stage 1 complete: haploid pool built for 31 genes.")
    print(f"Control haploids: {len(control_haploid_pool)}")
    print(f"Patient haploids: {len(patient_haploid_pool)}")
    print(f"File: {os.path.basename(haploid_save_path)}")
    print(f"**************************************************")

In [ ]:
import random
import pickle
import os

data_dir = "data_and_cache"
haploid_path = os.path.join(data_dir, "mTOR_HAPLOID_pool.pkl")

with open(haploid_path, "rb") as f:
    haploid_data = pickle.load(f)

control_pool = haploid_data["controls_haploid"]
patient_pool = haploid_data["patients_haploid"]
gene_names = haploid_data["gene_names"]

# Standard IUPAC ambiguity codes for heterozygous base pairs.
IUPAC_CODES = {
    ('A', 'G'): 'R', ('G', 'A'): 'R',
    ('C', 'T'): 'Y', ('T', 'C'): 'Y',
    ('A', 'C'): 'M', ('C', 'A'): 'M',
    ('G', 'T'): 'K', ('T', 'G'): 'K',
    ('C', 'G'): 'S', ('G', 'C'): 'S',
    ('A', 'T'): 'W', ('T', 'A'): 'W'
}

def merge_diploid_iupac(allele1_individual, allele2_individual):
    """Combine two haploid individuals into one diploid: matching bases pass
    through unchanged, mismatches are collapsed into an IUPAC ambiguity code."""
    iupac_individual_genes = []
    for g in range(len(allele1_individual)):
        seq1 = allele1_individual[g]
        seq2 = allele2_individual[g]
        iupac_gene_seq = []

        for b1, b2 in zip(seq1, seq2):
            if b1 == b2:
                iupac_gene_seq.append(b1)
            else:
                iupac_gene_seq.append(IUPAC_CODES.get((b1, b2), 'N'))
        iupac_individual_genes.append("".join(iupac_gene_seq))
    return iupac_individual_genes

def diploidize_population_iupac(haploid_pool, target_count):
    # Shuffle so pairing is random, then take consecutive pairs as the two
    # alleles of one diploid subject.
    diploid_population = []
    random.shuffle(haploid_pool)
    for i in range(target_count):
        allele_1 = haploid_pool[2*i]
        allele_2 = haploid_pool[2*i + 1]
        diploid_population.append(merge_diploid_iupac(allele_1, allele_2))
    return diploid_population

print("Sampling from the haploid pool and diploidizing with IUPAC codes...")
diploid_controls = diploidize_population_iupac(control_pool, 400)
diploid_patients = diploidize_population_iupac(patient_pool, 400)

final_dataset = {
    "controls": diploid_controls,
    "patients": diploid_patients,
    "gene_names": gene_names
}

pickle_save_path = os.path.join(data_dir, "mtor_data.pkl")
with open(pickle_save_path, "wb") as f:
    pickle.dump(final_dataset, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"\n**************************************************")
print(f"Stage 2 complete: population diploidized with IUPAC codes.")
print(f"Final dataset saved: {os.path.basename(pickle_save_path)}")
print(f"**************************************************")

In [ ]:
import pickle
import os

data_dir = "data_and_cache"
pickle_path = os.path.join(data_dir, "mtor_data.pkl")

print("Starting IUPAC diploidization and variation analysis...")
print(f"Reading file: {pickle_path}\n")

if not os.path.exists(pickle_path):
    raise FileNotFoundError("Final dataset .pkl file not found; run the diploidization cell first.")

with open(pickle_path, "rb") as f:
    data = pickle.load(f)

controls = data.get("controls", [])
patients = data.get("patients", [])
gene_names = data.get("gene_names", [])

# Sanity check: patients should show a higher IUPAC (heterozygote) rate than
# controls, since pathogenic sites are mutated more often for patients.
IUPAC_LETTERS = {'R', 'Y', 'M', 'K', 'S', 'W'}

control_iupac_count = 0
patient_iupac_count = 0
control_total_bases = 0
patient_total_bases = 0

print("======================================================================")
print("Group-level IUPAC heterozygote distribution")
print("======================================================================")

for individual in controls:
    for gene_seq in individual:
        control_total_bases += len(gene_seq)
        control_iupac_count += sum(1 for letter in gene_seq if letter in IUPAC_LETTERS)

for individual in patients:
    for gene_seq in individual:
        patient_total_bases += len(gene_seq)
        patient_iupac_count += sum(1 for letter in gene_seq if letter in IUPAC_LETTERS)

control_rate = (control_iupac_count / control_total_bases) * 100 if control_total_bases else 0
patient_rate = (patient_iupac_count / patient_total_bases) * 100 if patient_total_bases else 0

print(f"Control group:")
print(f"   - Total bases generated : {control_total_bases:,}")
print(f"   - IUPAC heterozygotes   : {control_iupac_count:,}")
print(f"   - Genomic het. rate     : {control_rate:.4f}%")
print(f"\nPatient group:")
print(f"   - Total bases generated : {patient_total_bases:,}")
print(f"   - IUPAC heterozygotes   : {patient_iupac_count:,}")
print(f"   - Genomic het. rate     : {patient_rate:.4f}%")
print("======================================================================\n")


print("======================================================================")
print("Per-gene IUPAC rate for the first 10 genes")
print("======================================================================")
print(f"{'No':<5} | {'Gene':<12} | {'Control IUPAC %':<18} | {'Patient IUPAC %':<15} | {'Status'}")
print("-" * 75)

for i in range(min(10, len(gene_names))):
    gene_name = gene_names[i]

    ctrl_gene_bases = sum(len(individual[i]) for individual in controls)
    ctrl_gene_iupac = sum(sum(1 for b in individual[i] if b in IUPAC_LETTERS) for individual in controls)
    ctrl_gene_pct = (ctrl_gene_iupac / ctrl_gene_bases) * 100 if ctrl_gene_bases else 0

    pat_gene_bases = sum(len(individual[i]) for individual in patients)
    pat_gene_iupac = sum(sum(1 for b in individual[i] if b in IUPAC_LETTERS) for individual in patients)
    pat_gene_pct = (pat_gene_iupac / pat_gene_bases) * 100 if pat_gene_bases else 0

    status = "OK (patient > control)" if pat_gene_pct > ctrl_gene_pct else "unexpected (equal/lower)"

    print(f"{i+1:<5} | {gene_name:<12} | {ctrl_gene_pct:<17.4f} | {pat_gene_pct:<15.4f} | {status}")

print("======================================================================")
print("Verification result")
print("======================================================================")
if patient_rate > control_rate and control_iupac_count > 0:
    print("Verified: the patient population's heterozygous mutation load exceeds the control group's.")
else:
    print("Unexpected result; check the haploid and diploidization steps.")
print("======================================================================")